# Coffee quality benchmark – inference pipeline

Standalone inference pipeline that uses the artifacts created by
`coffee-quality-pricing-benchmark_training_pipeline.ipynb` to predict the
`coffee_quality_score` of new coffee lots.

**Steps**

1. Configuration
2. Loading the model artifacts
3. Loading and validating the input data
4. Preprocessing with the fitted preprocessor
5. Prediction
6. Saving the predictions to `pipelines/output`
7. Optional evaluation (only if the input contains the target column)

The input CSV has to contain the feature columns of the training data.
Identifier, target and other extra columns are ignored; `record_id` is
kept in the output when present.

In [1]:
"""Inference pipeline for the coffee quality benchmark model."""

import json
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

## 1. Configuration

Paths are resolved relative to the `pipelines` folder. Set `INPUT_PATH`
to the CSV file to be scored.

In [2]:
PIPELINE_DIR = Path.cwd()
PROJECT_ROOT = PIPELINE_DIR.parent
ARTIFACTS_DIR = PIPELINE_DIR / "artifacts"
INPUT_PATH = PROJECT_ROOT / "data" / "coffee_inference_sample.csv"
OUTPUT_DIR = PIPELINE_DIR / "output"
OUTPUT_PATH = OUTPUT_DIR / "coffee_quality_predictions.csv"

MODEL_FILE_NAME = "coffee_quality_model.joblib"
PREPROCESSOR_FILE_NAME = "preprocessor.joblib"
SUMMARY_FILE_NAME = "model_summary.json"

ID_COLUMN = "record_id"
PREDICTION_COLUMN = "predicted_coffee_quality_score"

print(f"Artifacts dir: {ARTIFACTS_DIR}")
print(f"Input path:    {INPUT_PATH}")
print(f"Output path:   {OUTPUT_PATH}")

Artifacts dir: c:\D\Work\Ismeret\AI_ML_DL_TL\CUBIX_MLOPS_AIOPS\coffeePricePredictor\pipelines\artifacts
Input path:    c:\D\Work\Ismeret\AI_ML_DL_TL\CUBIX_MLOPS_AIOPS\coffeePricePredictor\data\coffee_inference_sample.csv
Output path:   c:\D\Work\Ismeret\AI_ML_DL_TL\CUBIX_MLOPS_AIOPS\coffeePricePredictor\pipelines\output\coffee_quality_predictions.csv


## 2. Loading the model artifacts

In [3]:
def load_artifacts(artifacts_dir: Path) -> tuple[Any, Any, dict[str, Any]]:
    """Load the model, the preprocessor and the training summary."""
    paths = [
        artifacts_dir / MODEL_FILE_NAME,
        artifacts_dir / PREPROCESSOR_FILE_NAME,
        artifacts_dir / SUMMARY_FILE_NAME,
    ]
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(
            f"Missing artifacts: {missing}. Run the training pipeline first."
        )

    model = joblib.load(paths[0])
    preprocessor = joblib.load(paths[1])
    summary = json.loads(paths[2].read_text(encoding="utf-8"))
    return model, preprocessor, summary


def get_column_groups(preprocessor: Any) -> tuple[list[str], list[str]]:
    """Return the numeric and categorical columns of the preprocessor."""
    numerical_columns: list[str] = []
    categorical_columns: list[str] = []
    for name, _, columns in preprocessor.transformers_:
        if name == "num":
            numerical_columns = list(columns)
        elif name == "cat":
            categorical_columns = list(columns)
    return numerical_columns, categorical_columns

In [4]:
model, preprocessor, summary = load_artifacts(ARTIFACTS_DIR)
target_column = summary["target_column"]
feature_columns = list(preprocessor.feature_names_in_)
numerical_columns, categorical_columns = get_column_groups(preprocessor)

print(f"Model:            {summary['best_model']}")
print(f"Trained at:       {summary.get('trained_at', 'n/a')}")
print(f"Target column:    {target_column}")
print(f"Feature columns:  {len(feature_columns)}")
print(f"Test metrics:     {summary['best_model_metrics']['test']}")

Model:            LightGBM
Trained at:       2026-10-21T13:17:50.522476+00:00
Target column:    coffee_quality_score
Feature columns:  31
Test metrics:     {'rmse': 1.6704806257554707, 'mae': 1.3235855525591058, 'r2': 0.884597760318714}


## 3. Loading and validating the input data

In [5]:
def load_input_data(input_path: Path) -> pd.DataFrame:
    """Load the CSV file to be scored."""
    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_path}")

    dataframe = pd.read_csv(input_path)
    if dataframe.empty:
        raise ValueError(f"Input file contains no rows: {input_path}")
    return dataframe


def prepare_inference_features(
    dataframe: pd.DataFrame,
    feature_columns: list[str],
    numerical_columns: list[str],
    categorical_columns: list[str],
) -> pd.DataFrame:
    """Validate the input columns and cast them to the training types."""
    missing = [
        column for column in feature_columns if column not in dataframe
    ]
    if missing:
        raise ValueError(f"Input is missing feature columns: {missing}")

    features = dataframe[feature_columns].copy()
    for column in numerical_columns:
        try:
            features[column] = pd.to_numeric(features[column])
        except (TypeError, ValueError) as error:
            raise ValueError(
                f"Column '{column}' must contain numeric values."
            ) from error
    for column in categorical_columns:
        series = features[column]
        features[column] = (
            series.astype(str).where(series.notna(), np.nan).astype(object)
        )
    return features

In [6]:
input_data = load_input_data(INPUT_PATH)
inference_features = prepare_inference_features(
    input_data,
    feature_columns,
    numerical_columns,
    categorical_columns,
)

print(f"Input shape: {input_data.shape}")
inference_features.head()

Input shape: (20, 32)


,harvest_season,harvest_calendar_year,harvest_month,country_of_origin,farm_type,farm_size_hectares,tree_age_years,altitude_mean_meters,species,avg_temp_c,...,traceability_level,certification,intended_buyer_segment,lot_size_kg,roast_level,acidity_score,sweetness_score,body_score,aroma_score,flavor_score
0,2021/22,2021,11,Colombia,Cooperative,15.60,21.0,2163.0,Arabica,12.5,...,Region,NaN,Premium,24372.0,Medium,8.17,7.88,7.17,7.84,8.15
1,2022/23,2023,5,Colombia,Smallholder,3.07,9.0,2079.0,Arabica,11.9,...,Country,NaN,Commodity,35448.0,Dark,7.42,7.55,7.86,8.57,7.75
2,2020/21,2020,12,Kenya,Cooperative,10.48,30.0,1973.0,Arabica,11.0,...,Cooperative,NaN,Premium,4810.0,Dark,8.33,7.54,8.94,8.77,7.86
3,2020/21,2020,12,Colombia,Cooperative,15.16,16.0,2003.0,Arabica,13.5,...,Country,Fair Trade,Commodity,25711.0,Dark,6.91,8.79,7.99,7.64,7.69
4,2022/23,2022,7,Indonesia,Cooperative,13.95,12.0,477.0,Robusta,23.3,...,Cooperative,Fair Trade,Commodity,16902.0,Medium,6.25,7.84,7.58,7.71,7.48


## 4. Preprocessing

In [7]:
processed_features = preprocessor.transform(inference_features)

print(f"Processed shape: {processed_features.shape}")

Processed shape: (20, 61)


## 5. Prediction

In [8]:
def build_output(
    dataframe: pd.DataFrame,
    predictions: np.ndarray,
) -> pd.DataFrame:
    """Combine the record identifiers with the predictions."""
    if ID_COLUMN in dataframe.columns:
        output = dataframe[[ID_COLUMN]].copy()
    else:
        output = pd.DataFrame(index=dataframe.index)
    output[PREDICTION_COLUMN] = predictions
    return output


predictions = model.predict(processed_features)
prediction_output = build_output(input_data, predictions)

prediction_output.head(20)

,record_id,predicted_coffee_quality_score
0,REC-0012093,79.424517
1,REC-0022018,80.373375
2,REC-0019926,82.220487
3,REC-0027317,79.489755
4,REC-0011839,70.713240
5,REC-0008363,80.289786
6,REC-0004153,75.480498
7,REC-0018134,74.202564
8,REC-0026646,71.659487
9,REC-0032575,72.475490


In [9]:
prediction_output[PREDICTION_COLUMN].describe()

count    20.000000
mean     76.795833
std       4.376945
min      69.080491
25%      73.928096
50%      75.854253
75%      80.310683
max      85.058600
Name: predicted_coffee_quality_score, dtype: float64

## 6. Saving the predictions

In [10]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
prediction_output.to_csv(OUTPUT_PATH, index=False)

print(f"{len(prediction_output)} predictions saved to {OUTPUT_PATH}")

20 predictions saved to c:\D\Work\Ismeret\AI_ML_DL_TL\CUBIX_MLOPS_AIOPS\coffeePricePredictor\pipelines\output\coffee_quality_predictions.csv


## 7. Optional evaluation

If the input data contains the true target values, the predictions are
evaluated with RMSE, MAE and R².

In [11]:
if target_column in input_data.columns:
    y_true = input_data[target_column]
    evaluation = {
        "rmse": float(np.sqrt(mean_squared_error(y_true, predictions))),
        "mae": float(mean_absolute_error(y_true, predictions)),
        "r2": float(r2_score(y_true, predictions)),
    }
    print(json.dumps(evaluation, indent=2))
else:
    print(f"'{target_column}' is not in the input data; evaluation skipped.")

'coffee_quality_score' is not in the input data; evaluation skipped.
